# PyTorch for LLMs 02: Models and Training Loops

> **What you finished last time:** named tensor axes and verified autograd against a manual derivative.
> **The gap this chapter closes:** a graph can compute gradients, but a reusable model needs registered parameters, a forward contract, and an explicit update lifecycle.
> **What this notebook delivers:** a saved linear model whose raw-tensor and \`nn.Module\` implementations are numerically equivalent.
> **Prerequisite for the next notebook:** \`artifacts/part02_linear_state.pt\` and \`artifacts/part02_config.json\`.

**Guiding question:** What does \`nn.Module\` add without changing the mathematics?

\`\`\`text
X                 @ W                 + b        = prediction
(example, feature) @ (feature, target) + (target,) = (example, target)
(96, 2)           @ (2, 1)            + (1,)      = (96, 1)
\`\`\`

\`\`\`mermaid
flowchart LR
    A["Visible tensor math"] --> B["Registered parameters"]
    B --> C["forward contract"]
    C --> D["loss scalar"]
    D --> E["backward"]
    E --> F["optimizer state"]
    F --> C
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports, deterministic data, and device ----------------------------------
import json
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_examples = 96
engine_load = torch.linspace(0.1, 1.0, n_examples)
ambient_temp = torch.linspace(-0.8, 0.9, n_examples).roll(17)
X = torch.stack([engine_load, ambient_temp], dim=1).to(device)
noise = 0.03 * torch.randn(n_examples, 1, device=device)
y = (1.8 * X[:, :1] - 0.65 * X[:, 1:2] + 0.4 + noise)
print(f"X={tuple(X.shape)} (example, feature); y={tuple(y.shape)} (example, target)")


## 1. Attempt 1: raw tensors can learn, but ownership is manual

**Predict:** Can autograd train two raw tensors? Yes. The complaint is not capability; it is organization and reusable state.


In [ ]:
# -- Create raw trainable tensors and a visible forward function ---------------
raw_W = torch.zeros(2, 1, device=device, requires_grad=True)
raw_b = torch.zeros(1, device=device, requires_grad=True)

def raw_forward(inputs, W, b):
    return inputs @ W + b

raw_initial = raw_forward(X, raw_W, raw_b)
print(f"Raw forward: {tuple(X.shape)} @ {tuple(raw_W.shape)} + {tuple(raw_b.shape)} -> {tuple(raw_initial.shape)}")


In [ ]:
# -- Train raw tensors with explicit gradient clearing and updates -------------
loss_function = nn.MSELoss()
raw_history = []
learning_rate = 0.12
for _ in range(180):
    prediction = raw_forward(X, raw_W, raw_b)
    loss = loss_function(prediction, y)
    loss.backward()
    with torch.no_grad():
        raw_W -= learning_rate * raw_W.grad
        raw_b -= learning_rate * raw_b.grad
    raw_W.grad.zero_()
    raw_b.grad.zero_()
    raw_history.append(loss.item())
print(f"Raw loss: {raw_history[0]:.6f} -> {raw_history[-1]:.6f}")
print(f"Raw parameters: W={raw_W.detach().cpu().ravel().tolist()}, b={raw_b.item():.3f}")


### That worked, but the model state is a loose collection

\`nn.Module\` registers parameters, moves them between devices, exposes a state dictionary, and standardizes calling the forward contract.


In [ ]:
# -- Encapsulate the same affine function in nn.Module ------------------------
class FuelLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, inputs):
        return self.linear(inputs)

torch.manual_seed(23)
model = FuelLinearModel().to(device)
with torch.no_grad():
    model.linear.weight.zero_()
    model.linear.bias.zero_()
print(f"Registered parameters: {list(dict(model.named_parameters()))}")


### Code walkthrough: FuelLinearModel

- The layer owns \`weight (target, feature)\` and \`bias (target,)\`.
- PyTorch stores \`nn.Linear.weight\` transposed relative to the visible \`X @ W\` notation.
- \`forward\` only states prediction math. Loss and optimization remain outside the model.


In [ ]:
# -- Verify the forward and loss contracts before optimization ----------------
module_prediction = model(X)
module_loss = loss_function(module_prediction, y)
assert module_prediction.shape == y.shape
print(f"Forward: X{tuple(X.shape)} -> prediction{tuple(module_prediction.shape)}")
print(f"Loss accepts prediction{tuple(module_prediction.shape)} and target{tuple(y.shape)} -> scalar{tuple(module_loss.shape)}")


## 2. The optimizer owns update state

**Predict:** If \`zero_grad()\` is omitted, will the next gradient replace the old one? No. It accumulates, as Part 01 measured.

\`\`\`mermaid
flowchart LR
    A["zero_grad"] --> B["prediction = model(X)"]
    B --> C["loss(prediction, y)"]
    C --> D["loss.backward"]
    D --> E["optimizer.step"]
    E --> A
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Isolate one module update and measure every state transition -------------
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
optimizer.zero_grad()
before = model.linear.weight.detach().clone()
prediction = model(X)
loss = loss_function(prediction, y)
loss.backward()
gradient_norm = model.linear.weight.grad.norm().item()
optimizer.step()
update_norm = (model.linear.weight.detach() - before).norm().item()
print(f"Gradient norm: {gradient_norm:.6f}")
print(f"Parameter update norm: {update_norm:.6f}")


In [ ]:
# -- Train the module with the same data, loss, and learning rate --------------
module_history = []
for _ in range(179):
    optimizer.zero_grad()
    prediction = model(X)
    loss = loss_function(prediction, y)
    loss.backward()
    optimizer.step()
    module_history.append(loss.item())
print(f"Module final loss: {module_history[-1]:.6f}")
print("The loop keeps model, loss, gradient, and update responsibilities separate.")


## 3. Prove raw and module implementations are equivalent

The raw model stores \`W(feature, target)\`. \`nn.Linear\` stores \`weight(target, feature)\`. Compare after the required transpose.


In [ ]:
# -- Compare learned parameters and predictions -------------------------------
module_W = model.linear.weight.detach().T
module_b = model.linear.bias.detach()
with torch.inference_mode():
    raw_prediction = raw_forward(X, raw_W, raw_b)
    module_prediction = model(X)
max_parameter_gap = max(
    (module_W - raw_W.detach()).abs().max().item(),
    (module_b - raw_b.detach()).abs().max().item(),
)
max_prediction_gap = (module_prediction - raw_prediction).abs().max().item()
print(f"Maximum parameter gap: {max_parameter_gap:.2e}")
print(f"Maximum prediction gap: {max_prediction_gap:.2e}")
assert max_parameter_gap < 1e-5 and max_prediction_gap < 1e-5


In [ ]:
# -- Plot both recorded optimization histories --------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(raw_history, label="raw tensors")
ax.plot([raw_history[0]] + module_history, linestyle="--", label="nn.Module")
ax.set(xlabel="update", ylabel="MSE", title="Same mathematics, different state ownership")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("Overlapping histories are evidence of implementation parity.")


## 4. Save the contract for the next fresh kernel

Notebooks do not share memory. Save learned state and the experiment facts needed to reconstruct the model.


In [ ]:
# -- Persist model state and explicit experiment configuration -----------------
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
state_path = artifact_dir / "part02_linear_state.pt"
config_path = artifact_dir / "part02_config.json"
torch.save(model.state_dict(), state_path)
config = {
    "seed": 23,
    "n_examples": n_examples,
    "input_features": 2,
    "output_features": 1,
    "learning_rate": learning_rate,
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Saved state: {state_path}")
print(f"Saved configuration: {config_path}")


### Your turn: change only the learning rate


In [ ]:
# -- CHANGE THIS: compare one optimizer setting for ten updates ---------------
YOUR_LEARNING_RATE = 0.03
torch.manual_seed(23)
trial = FuelLinearModel().to(device)
trial_optimizer = torch.optim.SGD(trial.parameters(), lr=YOUR_LEARNING_RATE)
trial_losses = []
for _ in range(10):
    trial_optimizer.zero_grad()
    trial_loss = loss_function(trial(X), y)
    trial_loss.backward()
    trial_optimizer.step()
    trial_losses.append(trial_loss.item())
print(f"Ten-update loss at lr={YOUR_LEARNING_RATE}: {trial_losses[0]:.4f} -> {trial_losses[-1]:.4f}")
assert YOUR_LEARNING_RATE > 0


## Scorecard and coverage ledger

| Tier | Covered here |
|---|---|
| Built and measured | raw tensor regression, \`nn.Module\`, forward/loss shapes, optimizer lifecycle, numerical parity, saved state |
| Explained or illustrated | parameter registration and optimizer state ownership |
| Named and out of scope | mini-batches, DataLoader, validation, resumable checkpoints, schedulers |

**Common failure modes:** target shape \`(N,)\` silently broadcasting against \`(N,1)\`, hiding loss inside the module, forgetting gradient clearing, and saving weights without reconstruction facts.

**Next:** Part 03 reloads this artifact from a fresh kernel, then forces batching, data loading, checkpoint, and hyperparameter contracts.
